In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025

# HDF5/netCDF4 su NFS + processi forkati puo' causare crash del pool (BrokenProcessPool)
# senza eccezione Python; disabilita il file locking HDF5 (fix standard per NFS).
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
import xarray as xr
import glob, os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams.update({'font.size': 22})
from scipy import stats
import matplotlib.patches as mpatches
import pandas as pd

import logging

import albedo_functions as af

In [ ]:
# Configurazione logging
log_file = "03-global_bias_tas_seasons.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),  # Write to file
    ]
)

In [ ]:
def global_average(dset):     
    lat_dim = 'lat' if 'lat' in dset.dims else 'latitude'
    lon_dim = 'lon' if 'lon' in dset.dims else 'longitude'
    
    weights = np.cos(np.deg2rad(dset[lat_dim]))
    weights.name = "weights"
    
    return dset.weighted(weights).mean((lat_dim, lon_dim))

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
era_var='2t'
SAVE_PATH = f'{WORK_DIR}/'
    
leads = [0, 1, 2, 3, 4]

In [ ]:
# Le funzioni di calcolo per il bias (lead singolo e combinazioni di lead year)
# stanno in un modulo importabile (bias_leadyears_lib.py): i processi spawn
# usati sotto (ogni task in un processo fresco, per evitare l'accumulo che
# faceva crashare il pool con i worker riusati) non vedono le funzioni
# definite nel notebook.
sys.path.insert(0, os.getcwd())
from bias_leadyears_lib import run_one_single, run_one


In [ ]:
# (vuota: la funzione di caricamento/calcolo per il lead singolo e' ora nel
# modulo bias_leadyears_lib.py, vedi cella precedente)


In [ ]:
%%time
import multiprocessing as mp

seasons = ['DJF', 'MAM', 'JJA', 'SON']
jobs_single = [(var, era_var, exp_ctrl, exp_sens, lead, season, SAVE_PATH)
               for season in seasons for lead in leads]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one_single, jobs_single), 1):
        print(f"  {msg}   {_i}/{len(jobs_single)}", flush=True)


In [ ]:
# La funzione di calcolo per le combinazioni di lead year sta in un modulo
# importabile (bias_leadyears_lib.py): i processi spawn usati sotto (ogni task
# in un processo fresco, per evitare l'accumulo che faceva crashare il pool con
# i worker riusati) non vedono le funzioni definite nel notebook.
sys.path.insert(0, os.getcwd())
from bias_leadyears_lib import run_one


In [ ]:
%%time
import multiprocessing as mp

seasons = ['DJF', 'MAM', 'JJA', 'SON']
jobs = [(var, era_var, exp_ctrl, exp_sens, y1, y2, season, SAVE_PATH)
        for season in seasons for y1 in range(5) for y2 in range(y1 + 1, 5)]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)
